# 松尾研 Physical AI｜実写17枚・極キャップ合成版
作成日時：2026/10/29 14:14 JST

添付原本17枚をそのまま収録し、側面11枚それぞれに上面・下面写真を極キャップとして合成した版である。上下面は同一個体と確認されていないため、合成上の補完である。Google Drive接続は不要。

**重要：** この新しい合成UVのPyBullet 10ケース実験は未検証。旧テクスチャの10/10成功を本版の成功として扱わない。戻り値0かつ個別10ケース合格を確認する。

In [ ]:
from pathlib import Path
import sys, os, shutil, zipfile, subprocess
from google.colab import files

print("CJK_FONT_PRECHECK")
font_candidates=[
    Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
    Path("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc")
]
if not any(p.is_file() for p in font_candidates):
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","fonts-noto-cjk"],check=True)
print("FONT_READY",next((str(p) for p in font_candidates if p.is_file()),"NOT_FOUND"))


In [ ]:
from pathlib import Path
import sys, os, shutil, zipfile, subprocess
from google.colab import files

exact=Path("/content/PAI_REAL17_POLAR_COMPOSITE_20261029.zip")
matches=sorted(Path("/content").glob("PAI_REAL17_POLAR_COMPOSITE_20261029*.zip"))
if exact.is_file():
    name=exact
elif matches:
    name=matches[-1]
else:
    print("ZIPを選択：PAI_REAL17_POLAR_COMPOSITE_20261029.zip")
    uploaded=files.upload()
    assert len(uploaded)==1, "ZIPを1つだけ選択してください"
    uploaded_name=next(iter(uploaded))
    assert uploaded_name.startswith("PAI_REAL17_POLAR_COMPOSITE_20261029") and uploaded_name.endswith(".zip"), "違うZIPです"
    name=Path("/content")/uploaded_name

assert name.is_file() and name.name.startswith("PAI_REAL17_POLAR_COMPOSITE_20261029") and name.suffix==".zip", "POLAR COMPOSITE ZIPを指定してください"
root=Path("/content/PAI_REAL17_POLAR_STAGE88")
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(name) as z:
    assert z.testzip() is None, "ZIP CRC破損"
    for n in z.namelist():
        part=Path(n)
        assert not part.is_absolute() and ".." not in part.parts, "安全でないZIPパス"
    z.extractall(root)

subprocess.run([sys.executable,str(root/"scripts/check_direct_source.py")],cwd=root,check=True)
print("REAL17_POLAR_COMPOSITE_READY",root,"ZIP_BYTES",name.stat().st_size,flush=True)


In [ ]:
cmd=[
    sys.executable,"-u",str(root/"run_realphoto_full10.py"),
    "--repo",str(root),
    "--out","/content/PAI_REAL17_POLAR_TEST_RESULTS",
    "--install"
]
rc=subprocess.run(cmd,cwd=root,check=False).returncode
print("REAL17_POLAR_PHYSICS_RETURN",rc,"0=10/10合格, 2=実験未達, 3=環境・検査エラー",flush=True)
result=Path("/content/PAI_G8_REAL17_FULL10_RESULT.zip")
if result.is_file():
    print("RESULT_READY",result,result.stat().st_size,flush=True)
    files.download(str(result))
else:
    print("RESULT_ZIP_NOT_CREATED; 上の実行ログを保存してください",flush=True)
